# Laboratorio 4 — Limpieza y Preparación de Datos
**Curso:** Minería de Datos (EIN132A25)
**Dataset:** Pokémon (Gen 1–6)

## Objetivos
- Detectar y tratar **valores faltantes**
- Identificar y manejar **outliers**
- Convertir variables categóricas (**encoding**)
- Aplicar **normalización y estandarización**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
df = pd.read_csv(url)
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
print("Tipos de datos:")
print(df.dtypes)
print("\nEstadísticas básicas:")
df.describe()

## 1. Valores faltantes

### Detección

In [ ]:
print('Valores faltantes por columna:')
print(df.isnull().sum())
print('\nPorcentaje de faltantes:')
print((df.isnull().mean() * 100).round(2))

In [ ]:
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Mapa de valores faltantes — Pokémon")
plt.show()

### Estrategias de tratamiento

La columna `Type 2` tiene ~48% de valores faltantes, ya que no todos los Pokémon tienen un segundo tipo. Esto **no es un error** en los datos — es información válida. Lo correcto es imputar con la categoría `'None'`.

In [ ]:
# Type 2 faltante → 'None' (pokémon mono-tipo)
df["Type 2"] = df["Type 2"].fillna("None")
print(f"Faltantes en 'Type 2' tras imputación: {df['Type 2'].isnull().sum()}")
print("\nDistribución de Type 2 (top 10):")
print(df["Type 2"].value_counts().head(10))

In [ ]:
print('Valores faltantes tras tratamiento:')
print(df.isnull().sum())

## 2. Outliers — Detección con IQR

Los Pokémon legendarios suelen tener estadísticas extremas. Vamos a analizar el `Total` de stats.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=df["Total"], ax=axes[0])
axes[0].set_title("Distribución de Total Stats — outliers")

sns.boxplot(x=df["Speed"], ax=axes[1])
axes[1].set_title("Distribución de Speed — outliers")

plt.tight_layout()
plt.show()

Q1 = df["Total"].quantile(0.25)
Q3 = df["Total"].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

outliers = df[df["Total"] > limite_superior]
print(f"Outliers detectados (Total > {limite_superior:.0f}): {len(outliers)}")
print(outliers[["Name", "Type 1", "Total", "Legendary"]].head(10))

In [ ]:
# Transformación logarítmica para reducir sesgo
df["Total_log"] = np.log1p(df["Total"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["Total"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Total Stats — original")
axes[1].hist(df["Total_log"], bins=30, color="salmon", edgecolor="white")
axes[1].set_title("Total Stats — log-transformado")
plt.tight_layout()
plt.show()

## 3. Encoding de variables categóricas

### Label Encoding — Type 1

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Type1_encoded"] = le.fit_transform(df["Type 1"])
print("Clases y su código:")
for i, cls in enumerate(le.classes_):
    print(f"  {i:2d} → {cls}")

In [ ]:
# Encoding booleano de Legendary
df["Legendary_int"] = df["Legendary"].astype(int)
print(f"Pokémon legendarios: {df['Legendary_int'].sum()} de {len(df)}")
print(df[["Name", "Legendary", "Legendary_int"]].head())

### One-Hot Encoding — Generation

In [ ]:
gen_dummies = pd.get_dummies(df["Generation"], prefix="Gen")
print(gen_dummies.head())

df = pd.concat([df, gen_dummies], axis=1)
print(f"Columnas tras OHE de Generation: {[c for c in df.columns if c.startswith('Gen_')]}")

## 4. Normalización y Estandarización

Las estadísticas de combate (HP, Ataque, Defensa, etc.) tienen escalas diferentes. Vamos a normalizarlas.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

stats_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

scaler_minmax = MinMaxScaler()
df[[f"{c}_norm" for c in stats_cols]] = scaler_minmax.fit_transform(df[stats_cols])

scaler_std = StandardScaler()
df[[f"{c}_std" for c in stats_cols]] = scaler_std.fit_transform(df[stats_cols])

print("Estadísticas — valores originales vs normalizados:")
df[["HP", "HP_norm", "HP_std", "Speed", "Speed_norm", "Speed_std"]].describe().round(3)

In [ ]:
# Visualización comparativa
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["Attack"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Attack — original")
axes[1].hist(df["Attack_norm"], bins=30, color="mediumseagreen", edgecolor="white")
axes[1].set_title("Attack — MinMax [0,1]")
axes[2].hist(df["Attack_std"], bins=30, color="salmon", edgecolor="white")
axes[2].set_title("Attack — StandardScaler")
plt.tight_layout()
plt.show()

## 5. Ejercicios

### Ejercicio 1 — Reporte de faltantes

In [ ]:
# Cargar el dataset original y reportar faltantes por columna
df_original = pd.read_csv(url)
faltantes = df_original.isnull().sum()
pct = (df_original.isnull().mean() * 100).round(2)
reporte = pd.DataFrame({"Faltantes": faltantes, "Porcentaje (%)": pct})
print(reporte[reporte["Faltantes"] > 0])

### Ejercicio 2 — Outliers en HP

In [ ]:
# Detectar outliers en HP con el método IQR y listar los pokémon afectados
Q1_hp = df_original["HP"].quantile(0.25)
Q3_hp = df_original["HP"].quantile(0.75)
IQR_hp = Q3_hp - Q1_hp
lim_sup_hp = Q3_hp + 1.5 * IQR_hp

outliers_hp = df_original[df_original["HP"] > lim_sup_hp]
print(f"Outliers en HP (>{lim_sup_hp:.0f}): {len(outliers_hp)}")
print(outliers_hp[["Name", "HP", "Type 1", "Legendary"]].sort_values("HP", ascending=False).head(10))

### Ejercicio 3 — Escalar las 6 estadísticas de combate

In [ ]:
# Escalar HP, Attack, Defense, Sp. Atk, Sp. Def, Speed con StandardScaler
# y verificar que la media sea ~0 y la desviación estándar sea ~1
from sklearn.preprocessing import StandardScaler

stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
sc = StandardScaler()
scaled = sc.fit_transform(df_original[stats])

print("Media por columna (debe ser ~0):")
print(dict(zip(stats, scaled.mean(axis=0).round(6))))
print("\nDesv. estándar (debe ser ~1):")
print(dict(zip(stats, scaled.std(axis=0).round(6))))

### Desafío — Pipeline de limpieza completo

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

def pipeline_limpieza_pokemon(df_raw):
    df_clean = df_raw.copy()

    # 1. Valores faltantes
    df_clean["Type 2"] = df_clean["Type 2"].fillna("None")

    # 2. Eliminar columnas no útiles
    df_clean = df_clean.drop(columns=["#", "Name"])

    # 3. Encoding
    le = LabelEncoder()
    df_clean["Type1_enc"] = le.fit_transform(df_clean["Type 1"])
    df_clean["Type2_enc"] = le.fit_transform(df_clean["Type 2"])
    df_clean["Legendary"] = df_clean["Legendary"].astype(int)
    df_clean = df_clean.drop(columns=["Type 1", "Type 2"])

    # 4. Scaling de estadísticas
    stats_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Total"]
    sc = StandardScaler()
    df_clean[stats_cols] = sc.fit_transform(df_clean[stats_cols])

    return df_clean

df_listo = pipeline_limpieza_pokemon(pd.read_csv(url))
print(f"Shape final: {df_listo.shape}")
print(f"Faltantes: {df_listo.isnull().sum().sum()}")
df_listo.head()